# Scraper — Observatorio Clima y Salud CDC Perú
**Proyecto Final — Diplomado AI UNI**

**Autor:** Alvaro Untiveros

Descarga automática de datos epidemiológicos por distrito desde:  
`https://app7.dge.gob.pe/maps2/shiny_observatorio_web/`

**Salida:** `{enfermedad}_consolidado_distrital.csv` — columnas en snake_case, ubigeo 6 dígitos.

In [2]:
#%pip install selenium webdriver-manager openpyxl charset_normalizer

## 1. Imports

In [4]:
import os, re, glob, time, logging
from pathlib import Path

import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver import ActionChains
from webdriver_manager.chrome import ChromeDriverManager

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%H:%M:%S'
)
log = logging.getLogger(__name__)
print('Imports OK')

Imports OK


## 2. Configuración
Rutas y enfermedades disponibles desde el link de CDC

In [ ]:
# ── Enfermedad ────────────────────────────────────────────────────────────────
# Opciones: "Dengue", "Malaria", "Leptospirosis", "Leishmaniasis",
#           "Ofidismo", "Febriles", "Neumonia", "SOB/Asma"
ENFERMEDAD = "Dengue"
TIPO_DX    = None   # "TOTAL", "CONFIRMADOS", "PROBABLES" o None (default)

# ── Rutas ─────────────────────────────────────────────────────────────────────
BASE_DIR     = Path(r'../')
DATA_RAW     = BASE_DIR / 'data' / 'raw'
DOWNLOAD_DIR = DATA_RAW / 'cdc_downloads' / ENFERMEDAD.lower().replace('/', '_')
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

# ── Selenium ──────────────────────────────────────────────────────────────────
BASE_URL = 'https://app7.dge.gob.pe/maps2/shiny_observatorio_web/'
HEADLESS = False
WAIT_SEC = 30

print(f'Enfermedad  : {ENFERMEDAD}')
print(f'Descarga en : {DOWNLOAD_DIR}')
print(f'CSV salida  : {DATA_RAW}')

Enfermedad  : Dengue
Descarga en : D:\Universidad\Alvaro\Proyecto UNI\UNI\Proyecto\data\raw\cdc_downloads\dengue
CSV salida  : D:\Universidad\Alvaro\Proyecto UNI\UNI\Proyecto\data\raw


## 3. Funciones auxiliares

In [8]:
def slugify(text: str) -> str:
    return re.sub(r'[^a-z0-9]+', '_', text.lower()).strip('_')

def make_driver(download_dir: Path, headless: bool) -> webdriver.Chrome:
    opts = webdriver.ChromeOptions()
    opts.add_experimental_option('prefs', {
        'download.default_directory': str(download_dir),
        'download.prompt_for_download': False,
        'plugins.always_open_xlsx_externally': True,
    })
    opts.add_argument('--disable-blink-features=AutomationControlled')
    opts.add_argument('--start-maximized')
    if headless:
        opts.add_argument('--headless=new')
        opts.add_argument('--window-size=1920,1080')
    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()), options=opts
    )

def wait_js_idle(driver, pause=0.8, max_tries=60) -> bool:
    for _ in range(max_tries):
        time.sleep(pause)
        try:
            busy = driver.find_elements(
                By.CSS_SELECTOR,
                'div.shiny-load-requests-waiting, div.shiny-busy'
            )
            if not any(e.is_displayed() for e in busy):
                return True
        except Exception:
            pass
    log.warning('wait_js_idle: tiempo máximo alcanzado')
    return False

def wait_download(download_dir: Path, before: set, timeout=180) -> list:
    t0 = time.time()
    while time.time() - t0 < timeout:
        new = set(os.listdir(download_dir)) - before
        complete = [f for f in new if not f.endswith(('.crdownload', '.tmp'))]
        if complete:
            time.sleep(1.5)
            return complete
        time.sleep(1)
    return []

def expand_accordion(driver, wait, panel_text: str) -> bool:
    try:
        xpath = (f"//button[contains(@class,'accordion-button') and "
                 f".//div[normalize-space()='{panel_text}']]")
        btn = wait.until(EC.element_to_be_clickable((By.XPATH, xpath)))
        if btn.get_attribute('aria-expanded') == 'false':
            driver.execute_script('arguments[0].scrollIntoView({block:"center"});', btn)
            time.sleep(0.4); btn.click(); time.sleep(1.2)
        return True
    except Exception as e:
        log.error(f'expand_accordion "{panel_text}": {e}'); return False

def set_slider_full(driver, wait) -> bool:
    try:
        sl = wait.until(EC.presence_of_element_located((By.ID, 'tendencias-sel_ano')))
        mn, mx = sl.get_attribute('data-min'), sl.get_attribute('data-max')
        driver.execute_script(
            f'$("#tendencias-sel_ano").data("ionRangeSlider").update({{from:{mn},to:{mx}}});'
        )
        log.info(f'Slider → {mn} a {mx}'); return True
    except Exception as e:
        log.error(f'set_slider_full: {e}'); return False

def find_input(driver, wait, label: str):
    try:
        xpath = f"//label[normalize-space()='{label}']/following-sibling::div//input"
        return wait.until(EC.visibility_of_element_located((By.XPATH, xpath)))
    except Exception:
        log.warning(f'find_input: no encontró "{label}"'); return None

def selectize_choose(driver, input_el, option: str, retries=3) -> bool:
    if not input_el: return False
    for attempt in range(retries):
        try:
            parent = input_el.find_element(
                By.XPATH, "./ancestor::div[contains(@class,'selectize-input')]"
            )
            driver.execute_script('arguments[0].click();', parent)
            time.sleep(0.4)
            input_el.send_keys(Keys.CONTROL + 'a'); input_el.send_keys(Keys.BACK_SPACE)
            time.sleep(0.2); input_el.send_keys(option); time.sleep(1.2)
            opt_xpath = (f"//div[contains(@class,'selectize-dropdown-content')]"
                         f"//div[normalize-space()='{option}']")
            WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable((By.XPATH, opt_xpath))
            ).click()
            wait_js_idle(driver); return True
        except Exception:
            log.debug(f'selectize_choose intento {attempt+1}/{retries} para "{option}"')
            ActionChains(driver).send_keys(Keys.ESCAPE).perform(); time.sleep(0.8)
    log.error(f'selectize_choose: falló para "{option}"'); return False

def get_selectize_options(driver, input_el, exclude=None) -> list:
    skip = {'seleccione', 'todos', '...', 'nacional'}
    if exclude: skip |= {e.lower() for e in exclude}
    opts = []
    if not input_el: return opts
    try:
        parent = input_el.find_element(
            By.XPATH, "./ancestor::div[contains(@class,'selectize-input')]"
        )
        driver.execute_script('arguments[0].click();', parent); time.sleep(0.8)
        items = driver.find_elements(
            By.XPATH,
            "//div[contains(@class,'selectize-dropdown') and "
            "not(contains(@style,'display: none'))]//div[@class='option']"
        )
        opts = [i.text.strip() for i in items
                if i.text.strip() and i.text.strip().lower() not in skip]
        ActionChains(driver).send_keys(Keys.ESCAPE).perform()
    except Exception as e:
        log.error(f'get_selectize_options: {e}')
    return list(dict.fromkeys(opts))

print('Funciones cargadas')

Funciones cargadas


## 4. Descarga por departamento

In [9]:
def run_scraper(enfermedad: str, tipo_dx, download_dir: Path) -> bool:
    driver = make_driver(download_dir, HEADLESS)
    wait   = WebDriverWait(driver, WAIT_SEC)

    try:
        log.info(f'▶ Iniciando: {enfermedad}')
        driver.get(BASE_URL); wait_js_idle(driver)

        wait.until(EC.element_to_be_clickable(
            (By.XPATH, "//a[@data-value='tendencias']")
        )).click()
        time.sleep(2); wait_js_idle(driver)

        # Enfermedad
        if not selectize_choose(driver, find_input(driver, wait, 'Enfermedad/episodio'), enfermedad):
            log.error('No se pudo seleccionar la enfermedad. Abortando.'); return False

        # Tipo diagnóstico (opcional)
        if tipo_dx:
            dx_input = find_input(driver, wait, 'Tipo de diagnóstico o forma clínica')
            if not selectize_choose(driver, dx_input, tipo_dx):
                log.warning(f'Tipo "{tipo_dx}" no disponible. Usando default.')

        # Período completo
        if not expand_accordion(driver, wait, 'Periodo de interés'): return False
        if not set_slider_full(driver, wait): return False

        # Departamentos
        if not expand_accordion(driver, wait, 'Lugar de interés'): return False
        dep_input     = find_input(driver, wait, 'Departamento')
        departamentos = get_selectize_options(driver, dep_input)
        if not departamentos:
            log.error('No se encontraron departamentos.'); return False
        log.info(f'{len(departamentos)} departamentos encontrados')

        slug_enf = slugify(enfermedad)
        exitosos, fallidos = [], []

        for dep in departamentos:
            log.info(f'  → {dep}')
            if not selectize_choose(driver, dep_input, dep):
                fallidos.append(dep); continue
            try:
                wait.until(EC.element_to_be_clickable(
                    (By.ID, 'tendencias-btn_zona'))
                ).click()
                wait_js_idle(driver, pause=2, max_tries=90)

                dl_btn = wait.until(EC.element_to_be_clickable(
                    (By.ID, 'tendencias-down_climasalud'))
                )
                before = set(os.listdir(download_dir))
                dl_btn.click()
                downloaded = wait_download(download_dir, before)

                if downloaded:
                    src = download_dir / downloaded[0]
                    dst = download_dir / f'{slug_enf}_{slugify(dep)}.xlsx'
                    if dst.exists(): dst.unlink()
                    src.rename(dst)
                    log.info(f'     ✓ {dst.name}'); exitosos.append(dep)
                else:
                    log.warning(f'     ✗ Timeout para {dep}'); fallidos.append(dep)

            except Exception as e:
                log.error(f'     Error en {dep}: {e}'); fallidos.append(dep)

        log.info(f'Descarga finalizada — OK: {len(exitosos)} | Fallidos: {len(fallidos)}')
        if fallidos: log.warning(f'Fallidos: {fallidos}')
        return len(exitosos) > 0

    except Exception:
        log.error('Error fatal', exc_info=True); return False
    finally:
        driver.quit()


run_scraper(ENFERMEDAD, TIPO_DX, DOWNLOAD_DIR)

02:06:45 [INFO] ====== WebDriver manager ======
02:06:46 [INFO] Get LATEST chromedriver version for google-chrome
02:06:46 [INFO] Get LATEST chromedriver version for google-chrome
02:06:46 [INFO] There is no [win64] chromedriver "146.0.7680.165" for browser google-chrome "146.0.7680" in cache
02:06:46 [INFO] Get LATEST chromedriver version for google-chrome
02:06:46 [INFO] WebDriver version 146.0.7680.165 selected
02:06:46 [INFO] Modern chrome version https://storage.googleapis.com/chrome-for-testing-public/146.0.7680.165/win32/chromedriver-win32.zip
02:06:46 [INFO] About to download new driver from https://storage.googleapis.com/chrome-for-testing-public/146.0.7680.165/win32/chromedriver-win32.zip
02:06:46 [INFO] Driver downloading response is 200
02:06:47 [INFO] Get LATEST chromedriver version for google-chrome
02:06:48 [INFO] Driver has been saved in cache [C:\Users\luana\.wdm\drivers\chromedriver\win64\146.0.7680.165]
02:06:51 [INFO] ▶ Iniciando: Dengue
02:07:02 [INFO] Slider → 201

True

## 5. Consolidación y limpieza
Une todos los `.xlsx`, estandariza columnas a snake_case y ubigeo a 6 dígitos.

In [14]:
COL_RENAME = {
    'UBIGEO': 'ubigeo', 'Ubigeo': 'ubigeo',
    'DEPARTAMENTO': 'departamento', 'Departamento': 'departamento',
    'PROVINCIA': 'provincia', 'Provincia': 'provincia',
    'DISTRITO': 'distrito', 'Distrito': 'distrito',
    'AÑO': 'ano', 'Año': 'ano', 'Ano': 'ano',
    'SEMANA': 'semana_epi', 'Semana': 'semana_epi',
    'CASOS': 'casos', 'Casos': 'casos',
    'Tipo caso': 'tipo_caso', 'TIPO CASO': 'tipo_caso',
    'tmean': 'tmean', 'tmax': 'tmax', 'tmin': 'tmin',
    'humr': 'humr', 'ptot': 'ptot',
    'tmean_clima': 'tmean_clima', 'tmax_clima': 'tmax_clima',
    'tmin_clima': 'tmin_clima', 'humr_clima': 'humr_clima', 'ptot_clima': 'ptot_clima',
}

def consolidar(download_dir: Path, enfermedad: str, out_dir: Path) -> pd.DataFrame:
    xlsx_files = sorted(download_dir.glob('*.xlsx'))
    if not xlsx_files:
        log.error(f'No hay .xlsx en {download_dir}'); return pd.DataFrame()

    dfs = []
    for fp in xlsx_files:
        try:
            xls   = pd.ExcelFile(fp)
            sheet = next((s for s in xls.sheet_names if s.lower() == 'distrito'), None)
            if sheet is None:
                log.warning(f'{fp.name}: sin hoja "Distrito" — {xls.sheet_names}'); continue
            dfs.append(pd.read_excel(xls, sheet_name=sheet))
        except Exception as e:
            log.warning(f'{fp.name}: {e}')

    if not dfs:
        log.error('No se pudo leer ningún archivo.'); return pd.DataFrame()

    df = pd.concat(dfs, ignore_index=True)
    df.rename(columns=COL_RENAME, inplace=True)

    if 'ubigeo' in df.columns:
        df['ubigeo'] = df['ubigeo'].astype(str).str.strip().str.zfill(6)

    if 'casos' in df.columns:
        df['casos'] = df['casos'].fillna(0).astype(int)

    out_path = out_dir / f'{slugify(enfermedad)}_consolidado_distrital.csv'
    df.to_csv(out_path, index=False, encoding='utf-8-sig')
    log.info(f'CSV -> {out_path} | {df.shape[0]:,} filas × {df.shape[1]} cols')
    return df


df_final = consolidar(DOWNLOAD_DIR, ENFERMEDAD, DATA_RAW)
df_final.head()

02:24:51 [INFO] CSV -> D:\Universidad\Alvaro\Proyecto UNI\UNI\Proyecto\data\raw\dengue_consolidado_distrital.csv | 908,980 filas × 18 cols


,departamento,provincia,distrito,ubigeo,tmean,tmax,tmin,humr,ptot,ano,semana_epi,tmean_clima,tmax_clima,tmin_clima,humr_clima,ptot_clima,tipo_caso,casos
0,AMAZONAS,BAGUA,ARAMANGO,010202,20.0,24.3,17.2,88.2,103.9,2018,1,20.6,24.6,17.6,81.1,21.6,NaN,0
1,AMAZONAS,BAGUA,BAGUA,010201,22.3,26.6,18.9,84.4,16.7,2018,1,23.3,27.5,19.9,75.6,7.8,TOTAL,7
2,AMAZONAS,BAGUA,COPALLIN,010203,20.0,23.7,16.7,85.4,46.0,2018,1,20.5,24.2,17.2,79.4,15.1,NaN,0
3,AMAZONAS,BAGUA,EL PARCO,010204,22.9,27.0,19.3,83.2,29.5,2018,1,23.9,27.8,20.4,75.2,10.7,NaN,0
4,AMAZONAS,BAGUA,IMAZA,010205,21.2,25.4,18.3,90.6,144.8,2018,1,21.6,25.6,18.4,86.7,24.3,NaN,0


## 6. Verificación rápida

In [15]:
if not df_final.empty:
    print('── Tipos ─────────────────────────────────')
    print(df_final.dtypes.to_string())
    print('\n── Rango temporal ────────────────────────')
    if 'ano' in df_final.columns:
        print(f"Años: {sorted(df_final['ano'].unique())}")

    print('\n── Nulos ─────────────────────────────────')
    nulos = df_final.isnull().sum()
    print(nulos[nulos > 0].to_string() or 'Sin nulos ✓')

    print('\n── Ubigeo ────────────────────────────────')
    if 'ubigeo' in df_final.columns:
        assert df_final['ubigeo'].str.len().eq(6).all(), 'X Ubigeos con longitud != 6'
        print(f"{df_final['ubigeo'].nunique()} distritos únicos — todos 6 dígitos Ok")

── Tipos ─────────────────────────────────
departamento     object
provincia        object
distrito         object
ubigeo           object
tmean           float64
tmax            float64
tmin            float64
humr            float64
ptot            float64
ano               int64
semana_epi        int64
tmean_clima     float64
tmax_clima      float64
tmin_clima      float64
humr_clima      float64
ptot_clima      float64
tipo_caso        object
casos             int32

── Rango temporal ────────────────────────
Años: [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]

── Nulos ─────────────────────────────────
tmean           91039
tmax            91039
tmin            91039
humr            98683
ptot            81228
tmean_clima     10810
tmax_clima      10810
tmin_clima      10810
humr_clima      10810
tipo_caso      850860

── Ubigeo ────────────────────────────────
1891 distritos únicos — todos 6 dígitos Ok
